# L200 · Agent evaluation — judge suite + AI Gateway

L100 introduced one LLM judge. L200 deepens the LLMOps spine into a **judge suite**: correctness and groundedness scored together over a small evaluation set, plus **AI Gateway** as the governance layer in front of the agent's model endpoint (rate limits and payload logging).

This notebook runs MLflow evaluation against the L200 OpenAI Agents SDK agent in `agent_server/agent.py`. Everything is parametrized; nothing is tied to one workspace.

### What you will do
1. Configure AI Gateway on the agent's serving endpoint (rate limit + payload logging).
2. Run the agent over a small AkzoNobel evaluation dataset.
3. Score each answer with **Correctness** and **Groundedness** judges, plus relevance and safety.
4. Read the per row scores and the aggregate.

> Note: all data is synthetic.

In [ ]:
%pip install "mlflow>=3.10.0" "openai-agents>=0.4.1" "databricks-openai>=0.13.0" "databricks-sdk>=0.70.0" nest_asyncio
dbutils.library.restartPython()

## 0. Parameters

The LLM endpoint and the judge model both come from widgets, so you can point evaluation at any endpoint you are entitled to.

In [ ]:
import os
import re

dbutils.widgets.text("llm_endpoint", "databricks-claude-sonnet-4-5", "Agent LLM endpoint")
dbutils.widgets.text("judge_endpoint", "databricks-claude-sonnet-4-5", "Judge model endpoint")
dbutils.widgets.text("catalog", "", "Unity Catalog (for the agent's tools)")

llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
judge_endpoint = dbutils.widgets.get("judge_endpoint").strip()
catalog = dbutils.widgets.get("catalog").strip() or spark.sql("SELECT current_catalog()").first()[0]

for name in (llm_endpoint, judge_endpoint):
    if not re.fullmatch(r"[A-Za-z0-9_.-]+", name):
        raise ValueError(f"Unsafe endpoint name: {name!r}.")

# The agent reads these from the environment when imported below.
os.environ["LLM_ENDPOINT"] = llm_endpoint
os.environ["AKZO_CATALOG"] = catalog
os.environ.setdefault("AKZO_TOOLS_SCHEMA", "akzo_tools")

print(f"Agent endpoint: {llm_endpoint}")
print(f"Judge endpoint: {judge_endpoint}")
print(f"Catalog:        {catalog}")

## 1. AI Gateway — governance in front of the model

AI Gateway sits on the serving endpoint and gives you rate limits, usage tracking, and payload (inference table) logging without changing agent code. Configuring it here makes the governance layer explicit: every call the agent makes to its model is rate limited and logged.

We set a conservative per endpoint rate limit and turn on payload logging. Adjust the numbers for your workspace. If you do not have permission to update the shared Foundation Model endpoint, skip this cell; evaluation still runs.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    AiGatewayConfig,
    AiGatewayRateLimit,
    AiGatewayRateLimitKey,
    AiGatewayRateLimitRenewalPeriod,
    AiGatewayInferenceTableConfig,
)

w = WorkspaceClient()

try:
    w.serving_endpoints.put_ai_gateway(
        name=llm_endpoint,
        rate_limits=[
            AiGatewayRateLimit(
                calls=100,
                key=AiGatewayRateLimitKey.ENDPOINT,
                renewal_period=AiGatewayRateLimitRenewalPeriod.MINUTE,
            )
        ],
        inference_table_config=AiGatewayInferenceTableConfig(
            enabled=True,
            catalog_name=catalog,
            schema_name="akzo_llmops",
            table_name_prefix="agent_gateway",
        ),
    )
    print("✓ AI Gateway configured: 100 calls/min, payload logging to akzo_llmops.")
except Exception as e:
    print("AI Gateway not configured (likely a permission or shared-endpoint limit).")
    print(f"  {e}")
    print("Evaluation still runs below.")

## 2. The evaluation dataset

A small set of AkzoNobel questions, each with the expected facts the answer should contain. Correctness compares the answer to `expected_facts`; Groundedness checks the answer is supported by the agent's retrieved context (its tool outputs).

In [ ]:
eval_dataset = [
    {
        "inputs": {"input": [{"role": "user", "content": "Which region had the lowest gross margin last quarter?"}]},
        "expectations": {"expected_facts": ["a region is named", "a margin figure is cited from the finance data"]},
    },
    {
        "inputs": {"input": [{"role": "user", "content": "What is the on time in full rate for our top supplier?"}]},
        "expectations": {"expected_facts": ["an OTIF percentage is given", "a supplier is named"]},
    },
    {
        "inputs": {"input": [{"role": "user", "content": "Summarize churn risk across our automotive accounts."}]},
        "expectations": {"expected_facts": ["automotive accounts are referenced", "a churn signal is described"]},
    },
    {
        "inputs": {"input": [{"role": "user", "content": "What is the capital of France?"}]},
        "expectations": {"expected_facts": ["the agent declines or states this is out of scope for coatings data"]},
    },
]
print(f"{len(eval_dataset)} evaluation rows.")

## 3. The predict function

We call the agent's registered `@invoke` function directly. It is async, so we wrap it for MLflow with `nest_asyncio`.

In [ ]:
import asyncio
import nest_asyncio
from mlflow.genai.agent_server import get_invoke_function
from mlflow.types.responses import ResponsesAgentRequest

# Importing the agent registers its @invoke function.
import agent_server.agent  # noqa: F401

invoke_fn = get_invoke_function()
assert invoke_fn is not None, "No @invoke-decorated function found in agent_server.agent."

nest_asyncio.apply()

def predict_fn(input, **kwargs):
    req = ResponsesAgentRequest(input=input)
    loop = asyncio.get_event_loop()
    response = loop.run_until_complete(invoke_fn(req))
    return response.model_dump()

## 4. Run the judge suite

`Correctness` and `Groundedness` are the two judges the plan calls for; we add `RelevanceToQuery` and `Safety` so you see a multi dimensional scorecard. Each judge returns a yes/no plus a rationale per row, and MLflow reports the aggregate.

In [ ]:
import mlflow
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery, Safety

# Groundedness: the answer must be supported by the agent's retrieved context.
groundedness = Guidelines(
    name="groundedness",
    guidelines=(
        "The response must only state facts that are supported by the tool outputs / "
        "retrieved context in the trace. If the data does not support a claim, the "
        "response should not make it."
    ),
)

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        Correctness(),
        groundedness,
        RelevanceToQuery(),
        Safety(),
    ],
)
results

## 5. Read the results

Open the run in the MLflow UI from the link above. You should see:
- A per row table with Correctness, groundedness, relevance, and safety, each with a yes/no and rationale.
- An aggregate pass rate per judge.
- Traces capturing the agent's tool calls (the UC function MCP calls), so you can see what context grounded each answer.

The deliberately out of scope question (capital of France) should score low on relevance unless the agent declines, which is the behavior the system prompt asks for.

If you enabled AI Gateway, the agent's model calls are also landing in `akzo_llmops.agent_gateway_*` inference tables, where you can audit payloads and enforce the rate limit.

### Next
Move up to `../L300-usecase/` for production monitoring, full UC lineage, and advanced regression gated evaluation.